Importing all the neccessary libraries

# World Bank Open Knowledge Repository — Corpus Builder

**Research context:** This notebook builds a structured literature corpus from the
[World Bank Open Knowledge Repository (OKR)](https://openknowledge.worldbank.org)
to support a research project on **fintech, financial inclusion, and poverty
reduction in Sub-Saharan Africa**, with a focus on South Sudan.

The corpus will later be used for keyword analysis, topic modelling, and
named-entity recognition on country and institution names.

**Why the World Bank OKR?**
The OKR hosts thousands of open-access reports, working papers, and journal
articles produced by the World Bank Group — directly relevant to development
economics research covering poverty, financial access, remittances, and digital inclusion.

**Why the API instead of HTML scraping?**
The OKR website is a JavaScript-rendered DSpace Angular app — BeautifulSoup
cannot parse it because the HTML is empty before JS runs. The site exposes a
clean public REST API returning structured JSON, which is more reliable and
richer than HTML scraping.

**Pipeline:**
1. Query the API with multiple thematic search terms and collect document IDs
2. Fetch full Dublin Core metadata for each document
3. Download a sample of PDFs
4. Export everything as a CSV

## Step 1: Import libraries

We need `requests` to call the API, `pandas` to organise results into a table,
`time` to pause between requests (polite scraping), and `os` to manage folders
for saved files.

In [ ]:
# Core libraries
# requests  : send HTTP calls to the API
# pandas    : organise results into a table
# time      : pause between requests (polite scraping)
# os        : manage folders for saved files

import os
import re
import time
import json
import random
from typing import Dict, Any, List, Optional, Tuple

# Networking + parsing
import requests

# Data handling
import pandas as pd

## Step 2: Configure the API connection and search parameters

All key settings are defined here as constants so the script is easy to adjust
without touching the logic below.

- `SEARCH_QUERIES`: list of thematic keywords — the script runs each one and merges results
- `PAGE_SIZE`: results per API page (API max is 100)
- `MAX_RESULTS_PER_QUERY`: cap per keyword to keep the sample manageable
- `MAX_PDFS`: how many PDFs to download as a sample

**Configuration**

In [ ]:
# Base URL of the World Bank OKR public REST API
API_BASE   = "https://openknowledge.worldbank.org/server/api"
SEARCH_URL = f"{API_BASE}/discover/search/objects"

# Multiple queries cover different facets of the research topic.
# Results from all queries will be merged and deduplicated by UUID.
SEARCH_QUERIES = [
    "financial inclusion developing countries",
    "poverty South Sudan Sub-Saharan Africa",
    "remittances fintech Africa",
    "digital financial services poverty",
]

PAGE_SIZE             = 20   # Results per API page
MAX_RESULTS_PER_QUERY = 40   # Maximum documents to collect per query keyword
MAX_PDFS              = 5    # Number of PDFs to download as a sample

# Polite session: identify ourselves to the server and request JSON responses
session = requests.Session()
session.headers.update({
    "Accept":     "application/json",
    "User-Agent": "Academic research - Jeremiah.Ater@SciencesPo.fr"
})

## Step 3: Phase 1: Collect document IDs across all search queries

For each keyword the script pages through the API using `?query=`, `?page=`,
and `?size=` URL parameters. We also have to pass `sort=dc.date.issued,DESC` to get
the most recent documents first.

Results are stored in a dictionary keyed by UUID — so any document that appears
in two different queries is automatically deduplicated.

**Phase 1: Collecting document IDs**

In [ ]:
# Dictionary keyed by UUID to automatically handle deduplication across queries
seen = {}

for query in SEARCH_QUERIES:
    print(f"\n--- Querying: '{query}' ---")
    page = 0

    while len([v for v in seen.values() if v["_query"] == query]) < MAX_RESULTS_PER_QUERY:
        resp = session.get(SEARCH_URL, params={
            "query": query,
            "page":  page,
            "size":  PAGE_SIZE,
            "sort":  "dc.date.issued,DESC"  # Most recent documents first
        })

        if resp.status_code != 200:
            print(f"  Stopped at page {page}: HTTP {resp.status_code}")
            break

        objects = resp.json()["_embedded"]["searchResult"]["_embedded"]["objects"]

        if not objects:
            print("  No more results.")
            break

        for obj in objects:
            item = obj.get("_embedded", {}).get("indexableObject", {})
            uuid = item.get("uuid")
            name = item.get("name")

            # Only store if we have a valid ID and title, and haven't seen it before
            if uuid and name and uuid not in seen:
                seen[uuid] = {
                    "uuid":       uuid,
                    "name":       name,
                    "handle":     item.get("handle"),
                    "public_url": f"https://openknowledge.worldbank.org/entities/publication/{uuid}",
                    "item_url":   item.get("_links", {}).get("self", {}).get("href"),
                    "_query":     query  # Track which query found this document
                }

        print(f"  Page {page}: {len(objects)} results — unique docs so far: {len(seen)}")
        page += 1
        time.sleep(1.5)  # Pause between pages to avoid overloading the server

candidates = list(seen.values())
print(f"\nTotal unique documents collected: {len(candidates)}")


--- Querying: 'financial inclusion developing countries' ---
  Page 0: 20 results — unique docs so far: 20
  Page 1: 20 results — unique docs so far: 40

--- Querying: 'poverty South Sudan Sub-Saharan Africa' ---
  Page 0: 10 results — unique docs so far: 50
  No more results.

--- Querying: 'remittances fintech Africa' ---
  Page 0: 1 results — unique docs so far: 51
  No more results.

--- Querying: 'digital financial services poverty' ---
  Page 0: 20 results — unique docs so far: 67
  Page 1: 20 results — unique docs so far: 87
  Page 2: 20 results — unique docs so far: 107

Total unique documents collected: 107


## Step 4: Phase 2: Enrich each document with full metadata

For each UUID we call `/core/items/{uuid}` to retrieve the full Dublin Core
metadata bundle. We then extract 10 fields: title, authors, date, document type,
subjects, abstract, DOI, language, country coverage, and JEL classification
codes (standard economics classification). A retry loop handles transient
network errors so no document is silently skipped.

**Phase 2: Fetch full metadata**

In [ ]:
def get_first(metadata, field):
    """Return the first value for a Dublin Core field, or None if missing."""
    entries = metadata.get(field, [])
    return entries[0].get("value") if entries else None

def get_all(metadata, field):
    """Return all values for a Dublin Core field joined by ' ; '."""
    return " ; ".join(
        e.get("value", "") for e in metadata.get(field, []) if e.get("value")
    )

enriched = []

for i, doc in enumerate(candidates):

    # Retry up to 3 times in case of a transient network error
    for attempt in range(3):
        try:
            resp = session.get(doc["item_url"], timeout=15)
            break
        except requests.RequestException:
            time.sleep(3)
    else:
        enriched.append(doc)  # Store doc without metadata if all attempts fail
        continue

    if resp.status_code != 200:
        enriched.append(doc)
        continue

    meta = resp.json().get("metadata", {})

    doc.update({
        "title":     get_first(meta, "dc.title"),
        "authors":   get_all(meta,   "dc.contributor.author"),
        "date":      get_first(meta, "dc.date.issued"),
        "type":      get_first(meta, "dc.type"),
        "subjects":  get_all(meta,   "dc.subject"),
        "abstract":  get_first(meta, "dc.description.abstract"),
        "doi":       get_first(meta, "dc.identifier.doi"),
        "language":  get_first(meta, "dc.language"),
        "country":   get_all(meta,   "dc.coverage.country"),
        "jel_codes": get_all(meta,   "dc.subject.jel"),  # Economics classification codes
    })

    enriched.append(doc)
    title_display = doc.get("title") or doc.get("name", "No title")  # fallback if title is None
    print(f"[{i+1}/{len(candidates)}] {title_display[:75]}")
    time.sleep(1.5)

[1/107] Formalizing Savings
[2/107] Livable Pacific Cities and Towns. Spotlight: Mapping Risk, Building Resilie
[3/107] Livable Pacific Cities and Towns. Spotlight: Tides of Change - Urban Resili
[4/107] Turning Vision Into Action: Advancing the Implementation of Climate Strateg
[5/107] Mauritius Country Climate and Development Report
[6/107] ID Meets Instant: Enabling Trusted, Inclusive Fast Payments through Digital
[7/107] Leveraging Digital Public Infrastructure for Payment Delivery and Inclusion
[8/107] The Rise and Regulation of Digital Credit: Lessons from Indonesia
[9/107] Investment in Emerging and Developing Economies
[10/107] Uganda Economic Update, December 2025: Cultivating Prosperity through Agro-
[11/107] Livable Pacific Cities and Towns: Urbanization Strategies for Resilience, S
[12/107] A Comparative Methodology for Measuring Supply-Side Domestic Payment Costs:
[13/107] Safeguarding Financial Inclusion During Crises: Lessons for Policymaking
[14/107] Do Behavioral Inter

## Step 5: Build the metadata table

We assemble all enriched documents into a pandas DataFrame, drop the internal
tracking columns used during collection, and reorder columns for readability.

**Build DataFrame**

In [ ]:
df = pd.DataFrame(enriched).drop(columns=["_query", "item_url"], errors="ignore")

# Reorder columns for readability
column_order = ["uuid", "handle", "title", "authors", "date", "type",
                "language", "country", "jel_codes", "subjects", "abstract",
                "doi", "public_url"]
df = df.reindex(columns=[c for c in column_order if c in df.columns])

# Add a simple sequential ID
df.insert(0, "id", range(1, len(df) + 1))

print(f"Corpus size: {len(df)} documents")
df.head(10)

Corpus size: 107 documents


,id,uuid,handle,title,authors,date,type,language,country,jel_codes,subjects,abstract,doi,public_url
0,1,235240d2-9c93-43e3-9214-ea7268a6f530,10986/44390,Formalizing Savings,"Melecky, Martin ; Singer, Dorothe",2026-02-24,Working Paper,English,,,FINANCIAL INCLUSION ; FORMAL AND INFORMAL SAVI...,This paper investigates the determinants of sa...,None,https://openknowledge.worldbank.org/entities/p...
1,2,c26d849c-0319-4e72-9f0e-d88cd584a84e,10986/44428,Livable Pacific Cities and Towns. Spotlight: M...,World Bank,2026-02-12,Report,English,,,PACIFIC CITIES AND TOWNS ; EARTHQUAKE HAZARD M...,Cities and towns in the Pacific region face si...,None,https://openknowledge.worldbank.org/entities/p...
2,3,6e36ce93-3468-4dae-9750-ba63949e1375,10986/44430,Livable Pacific Cities and Towns. Spotlight: T...,World Bank,2026-02-12,Report,English,,,URBAN GROWTH AND RESILIENCE ; PACIFIC ATOLL CI...,Pacific Island Countries (PICs) are experienci...,None,https://openknowledge.worldbank.org/entities/p...
3,4,8fa13bc6-7806-446b-97ca-99eb10cf9c00,10986/44431,Turning Vision Into Action: Advancing the Impl...,World Bank,2026-02-11,Report,English,,,CLIMATE INSTITUTIONAL RESILIENCE ; REGULATORY ...,"This assessment has four key objectives, which...",None,https://openknowledge.worldbank.org/entities/p...
4,5,7cd40926-75de-42af-a2ea-1a59fe22557b,10986/44325,Mauritius Country Climate and Development Report,World Bank Group,2026-02-10,Report,English,,,CLIMATE-RESILIENT TOURISM ; RESILIENT FISHERIE...,The Mauritius Country Climate and Development ...,None,https://openknowledge.worldbank.org/entities/p...
5,6,5d32dcd8-502f-4dba-9da0-2be838ae503e,10986/44316,"ID Meets Instant: Enabling Trusted, Inclusive ...",World Bank,2026-02-05,Report,English,,,DIGITAL ID ; FAST PAYMENT SYSTEMS (FPS) ; SING...,Digital public infrastructure has become a fou...,None,https://openknowledge.worldbank.org/entities/p...
6,7,96736e52-6069-45ec-bcf8-8733ce62144f,10986/44222,Leveraging Digital Public Infrastructure for P...,World Bank,2026-01-22,Report,English,,,INFRASTRUCTURE FINANCE ; VALUE-ADDED TAX (VAT)...,This report examines the design and implementa...,None,https://openknowledge.worldbank.org/entities/p...
7,8,804b60b1-27b0-4a27-b5a8-0f1688e1b00a,10986/44190,The Rise and Regulation of Digital Credit: Les...,"Alibhai, Salman ; Breza, Emily ; Kanz, Martin ...",2026-01-21,Working Paper,English,,,FINANCIAL INCLUSION ; DIGITAL FINANCE ; CREDIT...,This paper examines the rise of fintech lendin...,None,https://openknowledge.worldbank.org/entities/p...
8,9,77041a0b-332d-47ba-95b6-3a0ca98286f1,10986/44147,Investment in Emerging and Developing Economies,"Adarov, Amat ; Kose, M. Ayhan ; Vorisek, Dana",2026-01-07,Working Paper,English,,,INVESTMENT ; GROSSED FIXED CAPITAL FORMATION ;...,The world faces a pressing challenge to meet k...,None,https://openknowledge.worldbank.org/entities/p...
9,10,64b8a73a-094c-4345-ae59-62d616a9cd40,10986/44123,"Uganda Economic Update, December 2025: Cultiva...",World Bank,2025-12-24,Report,English,,,AGRO-INDUSTRIALIZATION ; CLIMATE VULNERABILITY...,This 26th Uganda Economic Update reports on r...,None,https://openknowledge.worldbank.org/entities/p...


## Step 6: Download a sample of PDFs

The API stores files inside "bundles". We navigate the chain
`/core/items/{uuid}/bundles → /bitstreams` to find the PDF link, then download it.
Only `MAX_PDFS` files are downloaded here to demonstrate the pipeline —
remove `.head(MAX_PDFS)` to retrieve all documents.

**Download PDFs**

In [ ]:
def get_pdf_url(uuid):
    """Follow the bundle → bitstream chain and return the download URL for the first PDF found."""
    try:
        bundles = session.get(f"{API_BASE}/core/items/{uuid}/bundles", timeout=15).json()
    except requests.RequestException:
        return None

    for bundle in bundles.get("_embedded", {}).get("bundles", []):
        if bundle.get("name") != "ORIGINAL":
            continue
        bs_href = bundle["_links"]["bitstreams"]["href"]
        try:
            bitstreams = session.get(bs_href, timeout=15).json()
        except requests.RequestException:
            continue
        for bs in bitstreams.get("_embedded", {}).get("bitstreams", []):
            if bs.get("name", "").endswith(".pdf"):
                return f"{API_BASE}/core/bitstreams/{bs['uuid']}/content"
    return None

# Create a folder to store the downloaded PDFs
os.makedirs("pdfs", exist_ok=True)

for i, row in df.head(MAX_PDFS).iterrows():
    pdf_url = get_pdf_url(row["uuid"])
    if not pdf_url:
        print(f"  No PDF found for: {row['title'][:60]}")
        continue

    content = session.get(pdf_url, timeout=60).content
    filename = f"pdfs/doc_{row['id']:02d}.pdf"
    with open(filename, "wb") as f:
        f.write(content)

    print(f"Downloaded: {filename}  ({len(content)//1024} KB)")
    time.sleep(5)  # Longer pause for large file downloads

Downloaded: pdfs/doc_01.pdf  (708 KB)
Downloaded: pdfs/doc_02.pdf  (12013 KB)
Downloaded: pdfs/doc_03.pdf  (70024 KB)
Downloaded: pdfs/doc_04.pdf  (7270 KB)
Downloaded: pdfs/doc_05.pdf  (40542 KB)


## Step 7: Export the corpus as a CSV file

The full metadata table is saved as a UTF-8 CSV. The `utf-8-sig` encoding
ensures correct display of accented characters in Excel. In Google Colab,
`files.download()` triggers a browser download automatically.

** Export CSV**

In [ ]:
df.to_csv("wb_corpus.csv", index=False, encoding="utf-8-sig")
print(f"Saved wb_corpus.csv — {len(df)} rows, {len(df.columns)} columns")

# Trigger download in Google Colab (comment out if running locally)
try:
    from google.colab import files
    files.download("wb_corpus.csv")
except ImportError:
    print("Not running in Colab — file saved to working directory.")

Saved wb_corpus.csv — 107 rows, 14 columns


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary and Methodological Note

This notebook built a structured academic corpus from the World Bank Open
Knowledge Repository (OKR) using its public REST API. The pipeline covered
four steps: collecting document IDs across multiple thematic search queries,
enriching each document with full Dublin Core metadata, downloading a sample
of PDFs, and exporting the complete dataset as a CSV file.

The final corpus covers 10 metadata fieldssuch as title, authors, date, document type, language, country coverage, JEL classification codes, subjects, abstract,
and DOI, across documents retrieved from four targeted queries related to
financial inclusion, poverty, remittances, and digital financial services in
Sub-Saharan Africa.

---

**Why BeautifulSoup was not used**

---

BeautifulSoup is a Python library designed to parse static HTML pages, it reads the raw HTML returned by a server and lets you navigate its structure
to extract content. This approach works well on traditional websites where
the content is embedded directly in the HTML source.

The World Bank OKR website is built with DSpace Angular, a modern
JavaScript-rendered framework. This means that when you request the page,
the server returns an almost empty HTML file, the actual content (titles,
abstracts, metadata) is only generated later by JavaScript running in the
browser. BeautifulSoup never sees this content because it only reads the
initial HTML response, not what the browser eventually renders.

Rather than working around this limitation using a browser automation tool
like Selenium (which simulates a full browser), the OKR conveniently exposes
a clean public REST API. This API returns structured JSON directly, with no
HTML parsing required. It is faster, more stable, and provides richer and
more precise metadata than any HTML scraping approach could. Using the API
was therefore not a shortcut, it was the most appropriate and robust method for this particular source.